# Lab 5D: Monte Carlo Tree Search and Stochastic Games

## Learning goals
By the end of this lab, students can:
- distinguish decision nodes from chance nodes;
- implement expectiminimax for a stochastic game;
- implement Monte Carlo Tree Search with UCB/UCT selection, expansion, simulation, and backpropagation;
- watch MCTS allocate visits across actions over time;
- compare sampled MCTS estimates with an exact expectiminimax baseline.

Source note: Chapter 5 motivates simulation-based search when full-width game search is too expensive; this lab makes that idea visual.

## Game used in this lab: Dice Race

Two players race to a target score. On each turn a player chooses an action; then chance determines the score change.

- `safe`: always gain 1 point.
- `risky`: 2/6 chance of 0, 3/6 chance of 2, 1/6 chance of 5.
- `sprint`: 3/6 chance of -1, 2/6 chance of 3, 1/6 chance of 6.

This gives us three node types:

1. **MAX/MIN decision nodes** choose actions.
2. **Chance nodes** average over dice outcomes.
3. **Terminal/cutoff nodes** return utility or evaluation.

In [ ]:
# Run this cell first.
# %pip install matplotlib ipywidgets

import math
import random
from functools import lru_cache
from collections import defaultdict

import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Rectangle, RegularPolygon

from IPython.display import display, HTML, clear_output
try:
    import ipywidgets as widgets
    WIDGETS_AVAILABLE = True
except Exception:
    WIDGETS_AVAILABLE = False
    print("ipywidgets is not available. Static snapshots will still work.")



def make_stepper(steps, draw_func, info_func=None, title="Search stepper"):
    """Small GUI stepper. Falls back to first/last static frames without ipywidgets."""
    if not steps:
        print("No steps to display.")
        return
    if WIDGETS_AVAILABLE:
        slider = widgets.IntSlider(value=0, min=0, max=len(steps)-1, step=1, description="step")
        play = widgets.Play(value=0, min=0, max=len(steps)-1, step=1, interval=700)
        widgets.jslink((play, "value"), (slider, "value"))
        out = widgets.Output()
        def render(change=None):
            with out:
                clear_output(wait=True)
                display(HTML(f"<h3>{title}</h3>"))
                if info_func:
                    display(HTML(info_func(steps[slider.value], slider.value)))
                draw_func(steps[slider.value], slider.value)
                plt.show()
        slider.observe(render, names="value")
        display(widgets.VBox([widgets.HBox([play, slider]), out]))
        render()
    else:
        for i in [0, len(steps)-1]:
            if info_func:
                display(HTML(info_func(steps[i], i)))
            draw_func(steps[i], i)
            plt.show()


TARGET = 12
ACTIONS = ("safe", "risky", "sprint")
OUTCOMES = {
    "safe": [(1.0, 1)],
    "risky": [(2/6, 0), (3/6, 2), (1/6, 5)],
    "sprint": [(3/6, -1), (2/6, 3), (1/6, 6)],
}


def terminal(state):
    p0, p1, turn = state
    return p0 >= TARGET or p1 >= TARGET


def utility(state):
    p0, p1, turn = state
    if p0 >= TARGET and p1 >= TARGET:
        return 0
    if p0 >= TARGET:
        return 1
    if p1 >= TARGET:
        return -1
    return 0


def evaluate(state):
    p0, p1, turn = state
    return max(-1, min(1, (p0 - p1) / TARGET))


def next_state(state, action, delta):
    p0, p1, turn = state
    if turn == 0:
        p0 = max(0, p0 + delta)
    else:
        p1 = max(0, p1 + delta)
    return (p0, p1, 1-turn)


def sample_next(state, action, rng):
    r = rng.random()
    total = 0
    for prob, delta in OUTCOMES[action]:
        total += prob
        if r <= total:
            return next_state(state, action, delta), delta
    prob, delta = OUTCOMES[action][-1]
    return next_state(state, action, delta), delta


def actor_name(turn):
    return "MAX" if turn == 0 else "MIN"

In [ ]:
@lru_cache(maxsize=None)
def expectiminimax(state, depth):
    """Exact finite-horizon value. MAX chooses max; MIN chooses min; chance nodes average."""
    if terminal(state):
        return utility(state), None
    if depth == 0:
        return evaluate(state), None
    p0, p1, turn = state
    action_values = []
    for action in ACTIONS:
        ev = 0
        for prob, delta in OUTCOMES[action]:
            ev += prob * expectiminimax(next_state(state, action, delta), depth-1)[0]
        action_values.append((action, ev))
    if turn == 0:
        return max(action_values, key=lambda x: x[1])[1], max(action_values, key=lambda x: x[1])[0]
    return min(action_values, key=lambda x: x[1])[1], min(action_values, key=lambda x: x[1])[0]


def action_values_exact(state, depth):
    rows = []
    for action in ACTIONS:
        ev = 0
        for prob, delta in OUTCOMES[action]:
            ev += prob * expectiminimax(next_state(state, action, delta), depth-1)[0]
        rows.append((action, ev))
    return rows

START_STATE = (4, 3, 0)
EXACT_DEPTH = 8
print("Start state:", START_STATE, "turn:", actor_name(START_STATE[2]))
print("Exact finite-horizon action values:", action_values_exact(START_STATE, EXACT_DEPTH))
print("Best expectiminimax:", expectiminimax(START_STATE, EXACT_DEPTH))

In [ ]:
def draw_expecti_tree(state, depth=2):
    """Draw a compact decision/chance tree for the first two plies."""
    nodes, edges = [], []
    def add(label, kind, value=None):
        nodes.append({"id": len(nodes), "label": label, "kind": kind, "value": value})
        return len(nodes) - 1
    root_id = add(f"{actor_name(state[2])}\n{state}", "decision", expectiminimax(state, depth)[0])
    for ai, action in enumerate(ACTIONS):
        chance_id = add(f"chance\n{action}", "chance")
        edges.append((root_id, chance_id, action))
        for prob, delta in OUTCOMES[action]:
            ns = next_state(state, action, delta)
            val = expectiminimax(ns, max(depth-1, 0))[0]
            child_id = add(f"{actor_name(ns[2])}\n{ns}\nv={val:.2f}", "decision", val)
            edges.append((chance_id, child_id, f"p={prob:.2f}, Δ={delta}"))
    # manual layout
    pos = {root_id: (0, 0)}
    chance_x = [-4, 0, 4]
    leaf_x = []
    for i, action in enumerate(ACTIONS):
        cx = chance_x[i]
        chance_id = 1 + i * 4  # one chance + three outcome nodes per action, given current OUTCOMES lengths except safe has 1; use robust below
    # robust layout from edges
    children = defaultdict(list)
    for p, c, lab in edges:
        children[p].append(c)
    for i, ch in enumerate(children[root_id]):
        pos[ch] = (chance_x[i], -1.4)
        out_children = children[ch]
        if len(out_children) == 1:
            xs = [chance_x[i]]
        else:
            xs = [chance_x[i] + (j-(len(out_children)-1)/2)*1.2 for j in range(len(out_children))]
        for x, leaf in zip(xs, out_children):
            pos[leaf] = (x, -2.8)
    fig, ax = plt.subplots(figsize=(12, 5))
    for p, c, label in edges:
        x1, y1 = pos[p]
        x2, y2 = pos[c]
        ax.plot([x1, x2], [y1, y2], color="#444444")
        ax.text((x1+x2)/2, (y1+y2)/2 + 0.05, label, fontsize=8, ha="center")
    for n in nodes:
        x, y = pos[n["id"]]
        if n["kind"] == "chance":
            ax.scatter([x], [y], s=1000, marker="D", c="#fef3bd", edgecolors="black", zorder=3)
        else:
            ax.scatter([x], [y], s=1300, marker="o", c="#d7e9ff", edgecolors="black", zorder=3)
        ax.text(x, y, n["label"], ha="center", va="center", fontsize=8, zorder=4)
    ax.set_title("Expectiminimax tree: decision nodes and chance nodes")
    ax.axis("off")
    ax.set_xlim(-5.5, 5.5)
    ax.set_ylim(-3.4, 0.6)
    plt.show()

draw_expecti_tree(START_STATE, depth=2)

In [ ]:
class MCTSNode:
    def __init__(self, state):
        self.state = state
        self.visits = 0
        self.total_value = 0.0
        self.children = {a: {} for a in ACTIONS}       # action -> next_state -> MCTSNode
        self.action_stats = {a: [0, 0.0] for a in ACTIONS}  # action -> [visits, total]
        self.tried_actions = set()

    @property
    def mean_value(self):
        return self.total_value / self.visits if self.visits else 0.0


def select_action(node, c=1.4):
    """UCT selection. MAX maximizes mean; MIN minimizes mean, with exploration."""
    turn = node.state[2]
    log_n = math.log(max(1, node.visits))
    best_action, best_score = None, -math.inf
    for action in ACTIONS:
        n, total = node.action_stats[action]
        if n == 0:
            return action
        mean_val = total / n
        exploit = mean_val if turn == 0 else -mean_val
        explore = c * math.sqrt(log_n / n)
        score = exploit + explore
        if score > best_score:
            best_action, best_score = action, score
    return best_action


def rollout(state, rng, limit=50):
    current = state
    for _ in range(limit):
        if terminal(current):
            return utility(current)
        action = rng.choice(ACTIONS)
        current, _ = sample_next(current, action, rng)
    return evaluate(current)


def mcts_iteration(root, rng, c=1.4):
    node = root
    visited_nodes = [node]
    path_edges = []
    while not terminal(node.state):
        untried = [a for a in ACTIONS if a not in node.tried_actions]
        if untried:
            action = rng.choice(untried)
            node.tried_actions.add(action)
        else:
            action = select_action(node, c=c)
        next_s, delta = sample_next(node.state, action, rng)
        if next_s not in node.children[action]:
            node.children[action][next_s] = MCTSNode(next_s)
            child = node.children[action][next_s]
            path_edges.append((node, action))
            visited_nodes.append(child)
            node = child
            break
        child = node.children[action][next_s]
        path_edges.append((node, action))
        node = child
        visited_nodes.append(node)
        if node.visits == 0:
            break
    value = rollout(node.state, rng)
    for n in visited_nodes:
        n.visits += 1
        n.total_value += value
    for parent, action in path_edges:
        parent.action_stats[action][0] += 1
        parent.action_stats[action][1] += value
    return value


def snapshot_root(root):
    snap = {"root_state": root.state, "root_visits": root.visits, "root_mean": root.mean_value, "actions": {}}
    for action in ACTIONS:
        n, total = root.action_stats[action]
        outcomes = []
        for s, child in root.children[action].items():
            outcomes.append({"state": s, "visits": child.visits, "mean": child.mean_value})
        snap["actions"][action] = {"visits": n, "mean": total/n if n else 0.0, "outcomes": outcomes}
    return snap


def run_mcts(state, iterations=800, seed=0, snapshot_points=None):
    if snapshot_points is None:
        snapshot_points = [1, 2, 5, 10, 25, 50, 100, 250, 500, iterations]
    rng = random.Random(seed)
    root = MCTSNode(state)
    snapshots = []
    history = []
    for i in range(1, iterations+1):
        mcts_iteration(root, rng)
        row = {"iteration": i}
        for action in ACTIONS:
            n, total = root.action_stats[action]
            row[action] = total/n if n else 0.0
            row[action + "_visits"] = n
        history.append(row)
        if i in snapshot_points:
            snapshots.append(snapshot_root(root))
    return root, snapshots, history

ROOT, SNAPSHOTS, HISTORY = run_mcts(START_STATE, iterations=800, seed=4)
print("Final root visits:", ROOT.visits)
for action in ACTIONS:
    n, total = ROOT.action_stats[action]
    print(action, "visits=", n, "mean=", round(total/n, 3) if n else None)

In [ ]:
def draw_mcts_snapshot(snap, idx=0):
    fig, ax = plt.subplots(figsize=(12, 5.5))
    ax.axis("off")
    root_x, root_y = 0, 0
    ax.scatter([root_x], [root_y], s=1800, c="#d7e9ff", edgecolors="black", zorder=3)
    ax.text(root_x, root_y, f"root\n{snap['root_state']}\nN={snap['root_visits']}", ha="center", va="center", fontsize=9)
    action_xs = [-4, 0, 4]
    for i, action in enumerate(ACTIONS):
        ax_x, ax_y = action_xs[i], -1.5
        stats = snap["actions"][action]
        width = 1 + 5 * (stats["visits"] / max(1, snap["root_visits"]))
        ax.plot([root_x, ax_x], [root_y, ax_y], lw=width, color="#555555", alpha=0.75)
        ax.scatter([ax_x], [ax_y], s=1350, c="#fef3bd", edgecolors="black", marker="D", zorder=3)
        ax.text(ax_x, ax_y, f"{action}\nN={stats['visits']}\nQ={stats['mean']:.2f}", ha="center", va="center", fontsize=9)
        outcomes = sorted(stats["outcomes"], key=lambda x: x["visits"], reverse=True)[:4]
        for j, out in enumerate(outcomes):
            ox = ax_x + (j - (len(outcomes)-1)/2) * 0.9
            oy = -3.0
            ow = 1 + 4 * (out["visits"] / max(1, stats["visits"]))
            ax.plot([ax_x, ox], [ax_y, oy], lw=ow, color="#888888", alpha=0.7)
            ax.scatter([ox], [oy], s=750, c="#e8f5e9", edgecolors="black", zorder=3)
            ax.text(ox, oy, f"{out['state']}\nN={out['visits']}\nQ={out['mean']:.2f}", ha="center", va="center", fontsize=7)
    ax.set_title(f"MCTS root after {snap['root_visits']} simulations: visit allocation and value estimates")
    ax.set_xlim(-5.5, 5.5)
    ax.set_ylim(-3.7, 0.8)
    plt.show()


def mcts_info_html(snap, idx):
    best = max(ACTIONS, key=lambda a: snap["actions"][a]["mean"] if snap["actions"][a]["visits"] else -999)
    return f"""
    <div style='font-family:Arial;line-height:1.35'>
    <b>Snapshot {idx+1} of {len(SNAPSHOTS)}</b><br>
    Simulations so far: <b>{snap['root_visits']}</b>. Current best sampled action for MAX: <b>{best}</b>.
    Edge thickness shows visit count. Q is average rollout utility for MAX.
    </div>
    """

make_stepper(SNAPSHOTS, draw_mcts_snapshot, mcts_info_html, title="Monte Carlo Tree Search growth")

In [ ]:
# Convergence plot: sampled MCTS estimates versus exact finite-horizon action values.
exact = dict(action_values_exact(START_STATE, EXACT_DEPTH))
fig, ax = plt.subplots(figsize=(9, 5))
for action in ACTIONS:
    xs = [h["iteration"] for h in HISTORY]
    ys = [h[action] for h in HISTORY]
    ax.plot(xs, ys, label=f"MCTS {action}")
    ax.axhline(exact[action], ls="--", alpha=0.5, label=f"exact {action}: {exact[action]:.2f}")
ax.set_xlabel("MCTS simulations")
ax.set_ylabel("estimated utility for MAX")
ax.set_title("MCTS estimates approach expectiminimax values with more samples")
ax.grid(True, alpha=0.25)
ax.legend(ncol=2, fontsize=8)
plt.show()

## Student practice

Change `START_STATE`, `TARGET`, or the outcome table. Which actions does MCTS explore more? How many simulations are needed before the sampled values stabilize?

In [ ]:
STUDENT_STATE = (8, 7, 0)
student_exact = action_values_exact(STUDENT_STATE, depth=6)
student_root, student_snaps, student_hist = run_mcts(STUDENT_STATE, iterations=400, seed=19)
print("Student exact values:", student_exact)
for action in ACTIONS:
    n, total = student_root.action_stats[action]
    print(action, "MCTS visits", n, "mean", round(total/n, 3) if n else None)
draw_mcts_snapshot(snapshot_root(student_root))